# Lecture 19

# Exploratory Data Analysis

## Week 7 Wednesday

## Miles Chen, PhD

# Import NHANES Data

We will work with data from NHANES - National Health and Nutrition Examination Survey.

https://wwwn.cdc.gov/Nchs/Nhanes/

We merge two tables from NHANES. 

1) The Demographic information table (DEMO_I) which contains information about the patients' sexes and ages.
2) The Body measures table (BMX_I) which contains information about the height, weight, and other measures.

There are many variables.

Data Dictionaries

https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm

https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BMX_I.htm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
bmx = pd.read_sas('../data/BMX_I.XPT')

In [ ]:
demo = pd.read_sas('../data/DEMO_I.XPT')

In [ ]:
merged = bmx.merge(right = demo, on = 'SEQN')

In [ ]:
merged

In [ ]:
merged.info()

## Do baby boys weigh more than baby girls on average?

The variable for the age in months is `RIDAGEMN`. 

https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#RIDAGEMN

The variable for gender is `RIAGENDR`

https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#RIAGENDR

The variable for the weight is `BMXWT`

https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BMX_I.htm#BMXWT

## Exploration with Seaborn
We can use Seaborn's lineplot to automatically average the values.

In [ ]:
# subset to select those with age in month 12 and under, and create a copy
babies = merged.loc[merged.RIDAGEMN <= 12].copy()

# Rename columns
babies = babies.rename(columns={
    'RIDAGEMN': 'age_months', 
    'BMXWT': 'weight_kg', 
    'RIAGENDR': 'gender'
})

In [ ]:
# map numeric categories to strings
babies['gender'] = babies['gender'].map({1.0: 'Boy', 2.0: 'Girl'})

In [ ]:
babies.loc[:,("age_months", "weight_kg", "gender")].head()

In [ ]:
babies.age_months = babies.age_months.astype(int) # convert age_months to integer
babies.loc[:,("age_months", "weight_kg", "gender")].head()

In [ ]:
sns.lineplot(data=babies, x='age_months', y='weight_kg', hue='gender')

In [ ]:
babies.loc[:,"gender"].value_counts()

In [ ]:
 # use pandas group_by and calculate mean
babies.groupby(['age_months', 'gender'])['weight_kg'].mean()

In [ ]:
summary_table = babies.groupby(['age_months', 'gender'])['weight_kg'].mean().unstack()
summary_table

In [ ]:
summary_table.plot() # very similar to the seaborn plot

### Comments:

The data shows that for each month, the mean weight of baby boys is greater than the mean weight of baby girls.

The mean weight of girls was never greater than the mean weight of boys.

## What is the relationship between height, leg length, and arm length? Does this relationship change with age?

The variable for age in years is `RIDAGEYR`
-https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/DEMO_I.htm#RIDAGEYR

The variables for height, leg length, and arm length are: `BMXHT`, `BMXLEG`, `BMXARML`

- https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BMX_I.htm#BMXHT
- https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BMX_I.htm#BMXLEG
- https://wwwn.cdc.gov/Nchs/Nhanes/2015-2016/BMX_I.htm#BMXARML

- We filter the data to all adults ( arbitrarily, I choose age 20 and up), who have all three measurements of height, upper leg length, and upper arm length.

In [ ]:
limbs = merged.loc[:, ['BMXHT', 'BMXLEG', 'BMXARML', 'RIDAGEYR']].copy()  # extract the needed columns
limbs = limbs.dropna()  # get rid of rows with missing values
adults = limbs.loc[limbs.RIDAGEYR >= 20,].copy() # subset to those with Age 20 and up
print(adults.info())

There are 5086 rows

Let's look at the correlation between:

   - standing height and upper leg length
   - standing height and upper arm length
   - upper leg length and upper arm length

In [ ]:
# Rename columns
adults = adults.rename(columns={
    'BMXHT': 'height', 
    'BMXLEG': 'leg_length',
    'BMXARML': 'arm_length',
    'RIDAGEYR': 'age_years'
})

In [ ]:
print(adults[['height', 'leg_length', 'arm_length']].corr())  # correlation values

I'll use `pandas.cut()` to make adult age groups by decade: i.e. adults aged 20-29.9, adults aged 30-39.9, ... adults aged 70-79.9, adults aged 80+ (7 groups total).

I'll add the column to the filtered data.

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.cut.html

In [ ]:
agegroups = pd.cut(adults.age_years, bins = np.arange(20, 99, 10), right = False)


In [ ]:
agegroups

In [ ]:
adults.loc[:,'age_category'] = agegroups

In [ ]:
adults

- For each age group, we calculate the mean of the three values.

In [ ]:
summary = adults.groupby('age_category', observed=True).mean().loc[:,['height','leg_length','arm_length']]
summary

In [ ]:
summary.index = np.arange(20,90,10)  # relabel the index
print(summary)

In [ ]:
summary.plot()

- Because leg length and arm length are similar in scale, but height is not, I add a fourth column - 'adjusted mean height' - which is the mean height minus 128 cm.

In [ ]:
summary['adjusted height'] = summary.height - 128
summary[['leg_length','arm_length','adjusted height']].plot()
plt.legend()

### Comments

There seems to be a very strong relationship between leg length and height. As people age, the leg length and height both seem to shrink over time.

Arm lenght, on the other hand does not seem to change very much over time. Arm length remains steady, possibly even getting longer over time.

## Is there a relationship between height or weight and income?

In [ ]:
income_weight = merged.loc[:, ['BMXHT', 'BMXWT', 'INDHHIN2', 'RIDAGEYR']].copy()  # extract the needed columns
income_weight = income_weight.dropna()  # get rid of rows with missing values
adults_inc_wt = income_weight.loc[income_weight.RIDAGEYR >= 20,].copy() # subset to those with Age 20 and up
print(adults_inc_wt.info())

In [ ]:
adults_inc_wt.groupby('INDHHIN2').mean()

In [ ]:
adults_inc_wt.loc[adults_inc_wt.INDHHIN2.isin([1,2,3,4,5,6,7,8,9,10,14,15]),:].groupby('INDHHIN2').mean()['BMXWT'].plot()

In [ ]:
adults_inc_wt.loc[adults_inc_wt.INDHHIN2.isin([1,2,3,4,5,6,7,8,9,10,14,15]),:].groupby('INDHHIN2').mean()['BMXHT'].plot()